# Checking the double charm categories

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import uproot
from numba import jit
import categories4
%matplotlib inline 
from particle import PDGID, Particle
from itertools import groupby

Welcome to JupyROOT 6.28/00


## Loading the file

In [2]:
def mygroupby(d, groupbycols):
    g = d.groupby(groupbycols).size().reset_index(name='count').sort_values([ 'count'], ascending=False).reset_index(drop=True)
    g["Percentage"] = g.apply(lambda row: 100 * row["count"]/d.shape[0], axis=1)
    g["cumulative %"] = g["Percentage"].cumsum(axis = 0)
    return g

## Loading the data for event type 23903000

In [3]:
datatype = "2012"
eventtype = "23903000"
polarity = "magdown"
sign = "rs"

In [4]:
%%time
def cache_tree(datatype, eventtype, polarity, sign):
    import ROOT
    data_prefix = "root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/final/ap_post_process/validated/"
    filename =  data_prefix + f"rds_finalnoBYsep_{datatype}_{eventtype}_{polarity}_{sign}.root"
    filename_categ =  f"rds_finalnoBYsepWithCateg_{datatype}_{eventtype}_{polarity}_{sign}.root"
    print(f"Loading DecayTree from {filename}")
    rdf = ROOT.RDataFrame("DecayTree", filename)
    rdf2 = categories4.add_categories_and_filter(rdf, apply_BDT_Iso_cut=True, apply_PIDK_cut=True)
    rdf2.Snapshot("DecayTree", filename_categ)
##cache_tree(datatype, eventtype, polarity, sign)
# Produced on pclhcb19

CPU times: user 7 µs, sys: 1e+03 ns, total: 8 µs
Wall time: 16.7 µs


In [5]:
def get_tree(datatype, eventtype, polarity, sign):
    filename = f"rds_finalnoBYsepWithCateg_{datatype}_{eventtype}_{polarity}_{sign}.root"
    print(f"Loading DecayTree from {filename}")
    dt = uproot.open(filename + ":DecayTree")
    return dt

In [6]:
dt = get_tree(datatype, eventtype, polarity, sign)

Loading DecayTree from rds_finalnoBYsepWithCateg_2012_23903000_magdown_rs.root


In [17]:
dt.arrays(["eventNumber"])

<Array [{eventNumber: 242597030}, ..., {...}] type='604310 * {eventNumber: ...'>

In [12]:
cols = [ b.name for b in dt.branches ]

In [13]:
[ c for c in cols if 'category' in c ]

['category']

In [9]:
categories_of_interest = [
    "Xc_signal_Ypis_nomatch_doubleCharm",
    "Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB",
    "Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB",]

In [10]:
key2name = { k:v for k,v in categories4.categories_map.items() if v in categories_of_interest }
name2key = { v:k for k, v in key2name.items()}
name2key

{'Xc_signal_Ypis_nomatch_doubleCharm': 7,
 'Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB': 14,
 'Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB': 15}

In [11]:
category = name2key['Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB']

# Extracting the ancestry information

extra_cols = [ "B_M", "category"]
arrs = dt.arrays(["p1_fromY_PANC_IDs", "p2_fromY_PANC_IDs", "p3_fromY_PANC_IDs",
                 "p1_fromY_PANC_Keys", "p2_fromY_PANC_Keys", "p3_fromY_PANC_Keys", ]
                 + extra_cols, f"category == {category}")
res = {}

@jit(nopython=True) 
def pprint(a):
    return ":".join([ str(int(p)) for p in a if p != 0])

@jit(nopython=True) 
def top_decay(a):
    return ":".join([ str(abs(int(p))) for p in a if p != 0 and abs(p) >= 500])

for i in range(1,4):
    tmp = arrs[f"p{i}_fromY_PANC_IDs"]
    res[f"p{i}_fromY_DECAY"] = [ pprint(p) for p in tmp ]        
    res[f"p{i}_fromY_TOP"] = [ top_decay(p) for p in tmp ]

    tmp = arrs[f"p{i}_fromY_PANC_Keys"]
    res[f"p{i}_fromY_Keys"] = [ pprint(p) for p in tmp ]

for c in extra_cols:
    res[c] = arrs[c]


# def create_df(dt, category):
#     # Extracting the ancestry information
    
#     extra_cols = [ "B_M", "category"]
#     arrs = dt.arrays(["p1_fromY_PANC_IDs", "p2_fromY_PANC_IDs", "p3_fromY_PANC_IDs",
#                      "p1_fromY_PANC_Keys", "p2_fromY_PANC_Keys", "p3_fromY_PANC_Keys", ]
#                      + extra_cols, f"category == {category}")
#     res = {}

#     @jit(nopython=True) 
#     def pprint(a):
#         return ":".join([ str(int(p)) for p in a if p != 0])

#     @jit(nopython=True) 
#     def top_decay(a):
#         return ":".join([ str(abs(int(p))) for p in a if p != 0 and abs(p) >= 500])

#     for i in range(1,4):
#         tmp = arrs[f"p{i}_fromY_PANC_IDs"]
#         res[f"p{i}_fromY_DECAY"] = [ pprint(p) for p in tmp ]        
#         res[f"p{i}_fromY_TOP"] = [ top_decay(p) for p in tmp ]

#         tmp = arrs[f"p{i}_fromY_PANC_Keys"]
#         res[f"p{i}_fromY_Keys"] = [ pprint(p) for p in tmp ]

#     for c in extra_cols:
#         res[c] = arrs[c]
        
#     common_ids, common_keys = find_common([arrs[f"p{i}_fromY_PANC_IDs"] for i in range(1,4)], 
#                                           [arrs[f"p{i}_fromY_PANC_Keys"] for i in range(1, 4)])
#     res["common_ids"] = pprint(common_ids)
#     res["common_keys"] = pprint(common_keys)
    
#     df = pd.DataFrame(res)
#     return df

In [12]:
def find_common(keys1, keys2, keys3, ids1):
    common_keys = []
    common_ids = []
    for i in range(len(keys1)):
        tmpk = []
        tmpid = []
        k1 = [ t for t in keys1[i] if t != 0.0 ]
        k2 = [ t for t in keys2[i] if t != 0.0 ]
        k3 = [ t for t in keys3[i] if t != 0.0 ]
        id1 = [ t for t in ids1[i] if t != 0.0 ]
        print(f"=== {k1} \n {k2} \n {k3} \n {id1}")
        for j in range(-1, - len(k1), -1):
            #print(f"=== Check {j} {k1[j]} {k2[j]} {k3[j]}")
            if k1[j] == k2[j] and k1[j] == k3[j]:
                #print(f"=== Common {k1[j]}")
                tmpk.append(k1[j])
                tmpid.append(id1[j])
            else:
                #print(f"=== breaking")
                break
        common_keys.append(pprint(list(reversed(tmpk))))
        common_ids.append(pprint(list(reversed(tmpid))))
    return common_ids, common_keys


#@jit(nopython=True) 
def _get_common3(k1, k2, k3):
    tmpk = []
    for j in range(-1, - len(k1), -1):
        if k1[j] == k2[j] and k1[j] == k3[j]:
            tmpk.append(k1[j])
        else:
            break
    return tmpk

def _get_common2(k1, k2):
    tmpk = []
    for j in range(-1, -1 * min(len(k1), len(k2)), -1):
        if k1[j] == k2[j]:
            tmpk.append(k1[j])
        else:
            break
    return tmpk




# def _identify_decay(k1, k2, k3, id1, id2, id3):
    
#     # Looking at the common part for the 3 decays
#     common_k = _get_common3(k1, k2, k3)
#     common_id = id1[-len(common_k):]
#     common_str = " -> ".join([ Particle.from_pdgid(p).name for p in reversed(common_id)])
#     print(common_str)
    
#     tmpk1 = k1[:-len(common_k)]
#     tmpk2 = k2[:-len(common_k)]
#     tmpk3 = k3[:-len(common_k)]
#     tmpid1 = id1[:-len(common_k)]
#     tmpid2 = id2[:-len(common_k)]
#     tmpid3 = id3[:-len(common_k)]
    
#     print(f"-- common: {common_id}")
#     print(f"--     id1: {tmpid1}")
#     print(f"--     id2: {tmpid2}")
#     print(f"--     id3: {tmpid3}")
    
#     # checking subdecay
#     decays = [(tmpk1, tmpid1), (tmpk2, tmpid2), (tmpk3, tmpid3)] 
#     subdecay = []
#     fromb = []
#     for key, group in groupby(decays, lambda x: x[-1][0]):
#         lg = list(group)
#         if len(lg) == 1:
#             fromb += lg
#         else:
#             subdecay.append(lg)
    
#     # dealing with the subdecay chain (normally only one pion, maybe resonnance ?)
#     frombstr = " -> ".join([ Particle.from_pdgid(p).name for p in reversed(fromb[0])])
    
#     # Printing the subdecay properly
#     if len(subdecay != 2):
#         raise Exception("Unhandled case, lenght of subdecay != 2")
#     sub_k = _get_common2(subdecay[0], subdecay[1])
#     #subdecaystr = "(" + " -> ".join([ Particle.from_pdgid(p).name for p in reversed(subdecay)]) + ")"
    
#     return " -> ".join([ common_str, frombstr, subdecaystr ])
    


def id2name(id):
    try:
        return Particle.from_pdgid(id).name
    except:
        return str(id)
    
import collections

class DecayData(collections.namedtuple('DecayData', [ 'keys', 'ids'])):
        __slots__ = ()
        def __str__(self):
            if len(self.ids) > 1:
                return "( " + " -> ".join([ id2name(p) for p in reversed(self.ids)]) + " )"
            elif len(self.ids) == 1:
                return id2name(self.ids[0])
            else:
                return ""
    

class Node(collections.namedtuple('Node',['parent','children'])):
        __slots__ = ()
        def __str__(self):
            ret = "[ " + str(self.parent) + " -> "
            ret += " ".join([str(c) for c in self.children])
            ret += " ]"
            return ret
           
def _subtract_parent(dd, common):
    return DecayData(dd.keys[:-len(common)], dd.ids[:-len(common)])


def _lists_to_graph3(dd1, dd2, dd3):
    
    common_k = _get_common3(dd1.keys, dd2.keys, dd3.keys)
    common_ids = dd1.ids[-len(common_k):]
    common = DecayData(common_k, common_ids)
    
    tmp1 = _subtract_parent(dd1, common_k)
    tmp2 = _subtract_parent(dd2, common_k)
    tmp3 = _subtract_parent(dd3, common_k)

    children = []
    
    decays = []
    if len(tmp1.keys) != 0:
        decays.append(tmp1)
    if len(tmp2.keys) != 0:
        decays.append(tmp2)
    if len(tmp3.keys) != 0:
        decays.append(tmp3)
        
    for key, group in groupby(decays, lambda x: x.keys[-1]):
        lg = list(group)
        if len(lg) == 1:
            children += lg
        elif len(lg) == 2:
            common_k = _get_common2(lg[0].keys, lg[1].keys)
            if len(common_k) > 0:
                children.append(_lists_to_graph2(*lg))
            else:
                children += lg

    return Node(common, children)


def _lists_to_graph2(dd1, dd2):
    
    common_k = _get_common2(dd1.keys, dd2.keys)
    if len(common_k) == 0:
        raise Exception("No common particles in the 2 decays")
    common_id = dd1.ids[-len(common_k):]
    common = DecayData(common_k, common_id)
  
    children = [ _subtract_parent(dd1, common_k), 
                 _subtract_parent(dd2, common_k) ]
    
    return Node(common, children)


def find_decay(keys1, keys2, keys3, ids1, ids2, ids3):
    decays = []
    for i in range(len(keys1)):
        tmpk = []
        tmpid = []
        k1 = [ t for t in keys1[i] if t != 0.0 ]
        k2 = [ t for t in keys2[i] if t != 0.0 ]
        k3 = [ t for t in keys3[i] if t != 0.0 ]
        id1 = [ t for t in ids1[i] if t != 0.0 ]
        id2 = [ t for t in ids2[i] if t != 0.0 ]
        id3 = [ t for t in ids3[i] if t != 0.0 ]
        
        #print(f"================= \n k1:{k1}\n k2:{k2}\n k3:{k3}")
        #print(f" id1:{id1}\n id2:{id2}\n id3:{k3}")    
        dec = _lists_to_graph3( DecayData(k1, id1), DecayData(k2, id2), DecayData(k3, id3))
        decays.append(str(dec))
        #print(dec)
    return decays
 
        
    
# def find_decay(keys1, keys2, keys3, ids1, ids2, ids3):
#     common_keys = []
#     common_ids = []
#     # Iterating on all candidates
#     for i in range(len(keys1)):
#         tmpk = []
#         tmpid = []
#         k1 = [ t for t in keys1[i] if t != 0.0 ]
#         k2 = [ t for t in keys2[i] if t != 0.0 ]
#         k3 = [ t for t in keys3[i] if t != 0.0 ]
#         id1 = [ t for t in ids1[i] if t != 0.0 ]
#         id2 = [ t for t in ids2[i] if t != 0.0 ]
#         id3 = [ t for t in ids3[i] if t != 0.0 ]
        
#         print(f"================= \n k1:{k1}\n k2:{k2}\n k3:{k3}")
#         print(f" id1:{id1}\n id2:{id2}\n id3:{k3}")    
#         dec = _identify_decay(k1, k2, k3, id1, id2, id3)
#         print(dec)
 


In [13]:
decays = find_decay(arrs[f"p1_fromY_PANC_Keys"],
                   arrs[f"p2_fromY_PANC_Keys"],
                   arrs[f"p3_fromY_PANC_Keys"],
                   arrs[f"p1_fromY_PANC_IDs"],
                   arrs[f"p2_fromY_PANC_IDs"],
                   arrs[f"p3_fromY_PANC_IDs"])

res["decay"] = decays
df = pd.DataFrame(res)

In [14]:
# common_ids, common_keys = find_common(arrs[f"p1_fromY_PANC_Keys"],
#                                       arrs[f"p2_fromY_PANC_Keys"],
#                                       arrs[f"p3_fromY_PANC_Keys"],
#                                       arrs[f"p1_fromY_PANC_IDs"])
# res["common_ids"] = common_ids
# res["common_keys"] = common_keys

# df = pd.DataFrame(res)


In [15]:
df['decay']

0        [ ( b~ -> B(s)*0 -> B(s)0 ) -> pi- [ D~0 -> pi...
1        [ ( b~ -> B+ ) -> ( D- -> e- ) pi+ ( D- -> K*(...
2        [ ( b -> B~0 ) -> [ D*(2010)+ -> ( D0 -> rho(7...
3        [ ( b~ -> B*0 -> B0 -> D*(2010)- ) -> pi- [ ( ...
4                [ ( b -> B~0 ) -> [ D0 -> pi+ pi- ] pi+ ]
                               ...                        
65459    [ ( b -> B~0 ) -> pi- [ D~0 -> ( K*(892)+ -> p...
65460    [ ( b~ -> B*0 ) -> ( B0 -> pi- ) [ B0 -> ( D~0...
65461    [ ( b~ -> B*0 ) -> [ B0 -> ( D~0 -> rho(770)- ...
65462    [ ( b -> B*~0 -> B~0 ) -> [ ( D0 -> eta ) -> p...
65463    [ ( b~ -> B*0 -> B0 ) -> pi- [ ( D~0 -> a(1)(1...
Name: decay, Length: 65464, dtype: object

In [16]:
# df['dec'] = df.apply(lambda row: row['common_ids'] + " / "
#          + row['p1_fromY_DECAY'].replace(row['common_ids'], '') + " / " 
#          + row['p1_fromY_DECAY'].replace(row['common_ids'], '') + " / " 
#          + row['p1_fromY_DECAY'].replace(row['common_ids'], ''), axis=1)

In [17]:
# df[ [ 'dec', 'p1_fromY_DECAY', 'p2_fromY_DECAY', 'p3_fromY_DECAY', 'common_ids']]

In [18]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 30)
g = mygroupby(df, 'decay')
g[:30]

,decay,count,Percentage,cumulative %
0,[ ( b -> B*~0 -> B~0 -> D*(2010)+ ) -> pi+ [ D0 -> pi- pi+ ] ],1443,2.204265,2.204265
1,[ ( b -> B*~0 -> B~0 -> D*(2010)+ ) -> [ D0 -> pi+ pi- ] pi+ ],1344,2.053037,4.257302
2,[ ( b~ -> B*0 -> B0 -> D*(2010)- ) -> [ D~0 -> pi- pi+ ] pi- ],1274,1.946108,6.203410
3,[ ( b~ -> B*0 -> B0 -> D*(2010)- ) -> pi- [ D~0 -> pi+ pi- ] ],1207,1.843761,8.047171
4,[ ( b -> B*~0 -> B~0 -> D*(2010)- ) -> pi- [ D~0 -> pi+ pi- ] ],654,0.999022,9.046193
5,[ ( b -> B*~0 -> B~0 -> D*(2010)- ) -> [ D~0 -> pi- pi+ ] pi- ],601,0.918062,9.964255
6,[ ( b~ -> B*0 -> B0 -> D*(2010)+ ) -> [ D0 -> pi+ pi- ] pi+ ],551,0.841684,10.805939
7,[ ( b -> B*~0 -> B~0 ) -> pi+ [ D0 -> pi- pi+ ] ],545,0.832519,11.638458
8,[ ( b~ -> B*0 -> B0 -> D*(2010)+ ) -> pi+ [ D0 -> pi- pi+ ] ],530,0.809605,12.448063
9,[ ( b -> B*~0 -> B~0 ) -> [ D0 -> pi+ pi- ] pi+ ],530,0.809605,13.257668


In [ ]:
%%time 
df = create_df(dt, name2key['Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB'])

In [ ]:
df[ [f"p{i}_fromY_Keys" for i in range(1,4)]]

In [ ]:
g23903000_hb = mygroupby(df23903000[df23903000.B_M > 4800 ], 'p1_fromY_TOP')
g23903000_lb = mygroupby(df23903000[df23903000.B_M < 4800 ], 'p1_fromY_TOP')